In [3]:
import sqlite3
import os

# 数据库路径
db_path = "/Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db"

# 检查文件是否存在
if not os.path.exists(db_path):
    print(f"❌ 文件不存在: {db_path}")
else:
    print(f"✅ 找到数据库: {db_path}")

# 建立连接
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 获取所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"\n📋 共 {len(tables)} 张表：")
for t in tables:
    print(f"   - {t[0]}")

✅ 找到数据库: /Users/jamesl/Library/CloudStorage/GoogleDrive-nxtagentimes@gmail.com/我的云端硬盘/6. CMI/数据集/cmidata.db

📋 共 2 张表：
   - purchase_orders
   - ar_aging


# 📊 CMI 采购订单数据分析

本 Notebook 用于分析 CMI的采购订单数据，数据来源为本地 SQLite 数据库 `cmidata.db`。

## 分析流程

1. **数据库连接** — 加载数据库，查看所有表
2. **表结构探索** — 浏览各表字段定义与数据量
3. **金额字段识别** — 找出与收入/费用相关的字段
4. **供应商收入统计** — 按供应商汇总订单数与总金额
5. **供应商详情查询** — 查询指定供应商的承办人与订单明细

## 1. 表结构探索

浏览数据库中所有表的字段定义（名称、类型、是否主键等）和行数，帮助理解数据结构。

In [4]:

import pandas as pd

# 显示每张表的字段定义
for (table_name,) in tables:
    print(f"\n{'='*60}")
    print(f"📊 表: {table_name}")
    print(f"{'='*60}")

    # PRAGMA table_info 返回：列序号, 列名, 数据类型, 是否非空, 默认值, 是否主键
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()

    # 用 DataFrame 展示，更清晰
    df_cols = pd.DataFrame(columns, columns=["序号", "列名", "类型", "非空", "默认值", "主键"])
    df_cols["主键"] = df_cols["主键"].map({1: "✅", 0: ""})
    df_cols["非空"] = df_cols["非空"].map({1: "✅", 0: ""})
    display(df_cols[["列名", "类型", "非空", "主键"]])

    # 同时显示行数
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"   行数: {count}")



📊 表: purchase_orders


,列名,类型,非空,主键
0,采购订单编号,TEXT,,
1,IBOSS产品类型,TEXT,,
2,订单状态,TEXT,,
3,承办人,TEXT,,
4,承办部门,TEXT,,
...,...,...,...,...
80,是否涉及特殊审批,REAL,,
81,特殊审批单号,REAL,,
82,实际取消/终止日期,REAL,,
83,是否有付款需求,REAL,,


   行数: 802

📊 表: ar_aging


,列名,类型,非空,主键
0,customer,TEXT,,
1,currency,TEXT,,
2,aging_0,REAL,,
3,aging_30,REAL,,
4,aging_90,REAL,,
5,aging_180,REAL,,
6,aging_360,REAL,,
7,aging_1y,REAL,,
8,credit_note,REAL,,
9,grand_total,REAL,,


   行数: 265


## 2. 金额相关字段识别

自动扫描 `purchase_orders` 表中包含「金额」「费用」「价格」等关键词的字段，便于确定后续分析所使用的金额列。

In [4]:
# 查找跟金额/收入相关的字段
cursor.execute("PRAGMA table_info(purchase_orders)")
all_cols = [row[1] for row in cursor.fetchall()]
money_cols = [c for c in all_cols if any(k in c for k in ["金额", "收入", "费用", "价格", "价", "合计", "总", "amount", "cost", "revenue", "price", "金额"])]
print("💰 包含金额/费用相关的字段：")
for c in money_cols:
    print(f"   - {c}")

💰 包含金额/费用相关的字段：
   - 报价单号
   - 采购订单总金额（港币）
   - NRC里程碑金额(原币不含税)
   - 突发流量价格


## 3. 供应商收入统计

按供应商汇总订单数与总金额（HKD），并列出各供应商提供的服务类型。同一订单编号的多行明细会先去重，避免金额重复计算。

In [5]:
# ⚠️ 同一订单编号在表中有多行明细，必须先按订单去重，否则金额会被重复计算
df_orders = pd.read_sql_query("""
    SELECT DISTINCT
        `采购订单编号`,
        `供应商名称`,
        `采购订单总金额（港币）` AS 订单金额,
        `Service Type`
    FROM purchase_orders
    WHERE `供应商名称` IS NOT NULL AND `供应商名称` != ''
""", conn)

print(f"📋 去重后订单数: {len(df_orders)}（原表有大量重复行）")

# 用 Pandas 聚合：按供应商统计金额 + 收集服务类型列表
df_revenue = df_orders.groupby("供应商名称", as_index=False).agg(
    订单数=("采购订单编号", "nunique"),
    总金额_HKD=("订单金额", "sum"),
    服务类型列表=("Service Type", lambda x: sorted(set(s for s in x if s and str(s).strip())))
)

# 服务类型格式化为逗号分隔的字符串，空值显示为 "未标注"
df_revenue["服务类型"] = df_revenue["服务类型列表"].apply(
    lambda lst: ", ".join(lst) if lst else "未标注"
)

# 排序
df_revenue = df_revenue.sort_values("总金额_HKD", ascending=False).reset_index(drop=True)

# 格式化金额显示
total_hkd = df_revenue["总金额_HKD"].sum()
df_revenue["总金额_HKD_display"] = df_revenue["总金额_HKD"].round(2).apply(lambda x: f"{x:,.2f}")

print(f"\n📊 供应商收入统计（共 {len(df_revenue)} 家供应商）")
print(f"   总计订单数: {df_revenue['订单数'].sum()}")
print(f"   总金额 (HKD): {total_hkd:,.2f}")

# 显示结果（含服务类型列）
display(df_revenue[["供应商名称", "订单数", "总金额_HKD_display", "服务类型"]].rename(
    columns={"总金额_HKD_display": "总金额 (HKD)"}
))

📋 去重后订单数: 251（原表有大量重复行）

📊 供应商收入统计（共 59 家供应商）
   总计订单数: 234
   总金额 (HKD): 37,084,467.22


,供应商名称,订单数,总金额 (HKD),服务类型
0,Smart Vic Limited,43,"5,912,123.31","其他, 现场集成服务, 集成实施服务"
1,曉通網路科技（香港）有限公司,25,"2,653,217.63","AAS CPE, AAS Other, CPE, 现场集成服务, 集成实施服务"
2,LINKCIRCLE HONGKONG COMPANY LIMITED,6,"2,448,217.92","Local Loop, Other"
3,METRO INTEGRATED TECH INC.,3,"1,826,037.95","ICT终端及设备, 现场集成服务, 集成实施服务"
4,Beijing Sinonet Information Technology Limited,1,"1,676,303.35",Port Sec
5,Digital Plan System,1,"1,617,464.58","ICT终端及设备, 集成实施服务"
6,"GUOLI Holdings Co., Limited",9,"1,591,610.60","CPE, Port Sec"
7,SICITEL LIMITED,1,"1,488,466.44",Port Sec
8,台灣固網股份有限公司,3,"1,398,263.34","Local Loop, Port Sec-IP"
9,FPT Telecom,1,"1,199,828.28",Local Loop


## 4. 供应商详情查询

查询指定供应商的详细订单信息，包括承办人、电路编号、服务类型和订单状态。修改顶部的 `supplier_name` 变量即可切换查询任意供应商（支持模糊匹配）。

In [7]:
#
supplier_name = "曉通"  # 👈 修改这里，改成任意供应商名称（支持模糊匹配）

# 查询指定供应商的承办人信息
df_supplier = pd.read_sql_query("""
    SELECT DISTINCT
        `采购订单编号`,
        `承办人`,
        `供应商名称`,
        `采购订单总金额（港币）` AS 订单金额,
        `客户电路编号`,
        `Service Type`,
        `订单状态`
    FROM purchase_orders
    WHERE `供应商名称` LIKE ?
""", conn, params=[f"%{supplier_name}%"])

print(f"🔍 「{supplier_name}」相关记录数: {len(df_supplier)}")
print(f"💰 总收入 (HKD): {df_supplier['订单金额'].sum():,.2f}")
print(f"📋 承办人: {df_supplier['承办人'].unique().tolist()}")

# 完整展示
display(df_supplier)

🔍 「曉通」相关记录数: 25
💰 总收入 (HKD): 2,653,217.63
📋 承办人: ['王千千', '吳致瑋', '黄菁', '刘炼']


,采购订单编号,承办人,供应商名称,订单金额,客户电路编号,Service Type,订单状态
0,CMI-PO-202605-01471,王千千,曉通網路科技（香港）有限公司,54998.33,NaN,AAS Other,审批通过
1,CMI-PO-202605-00345,吳致瑋,曉通網路科技（香港）有限公司,16451.47,FLE-0128,CPE,审批通过
2,CMI-PO-202604-01183,吳致瑋,曉通網路科技（香港）有限公司,48684.59,JABI0026,CPE,审批通过
3,CMI-PO-202604-00834,黄菁,曉通網路科技（香港）有限公司,6955.91,BRII0006,集成实施服务,审批通过
4,CMI-PO-202603-01548,王千千,曉通網路科技（香港）有限公司,1124446.24,NaN,AAS CPE,审批通过
5,CMI-PO-202603-01390,黄菁,曉通網路科技（香港）有限公司,95673.60,SEMI0016,现场集成服务,审批通过
6,CMI-PO-202603-01389,黄菁,曉通網路科技（香港）有限公司,95673.60,SEMI0015,现场集成服务,审批通过
7,CMI-PO-202603-01388,黄菁,曉通網路科技（香港）有限公司,95673.60,SEMI0014,现场集成服务,审批通过
8,CMI-PO-202603-01387,黄菁,曉通網路科技（香港）有限公司,92859.67,SEMI0013,现场集成服务,审批通过
9,CMI-PO-202603-01386,黄菁,曉通網路科技（香港）有限公司,92859.67,SEMI0012,现场集成服务,审批通过
